In [1]:
#!pip install transformers

#!pip install "transformers[torch]"

In [1]:
import pandas as pd
from transformers import T5Tokenizer , Trainer , TrainingArguments , T5ForConditionalGeneration

# import data file

In [3]:
train_data = pd.read_csv(r"DataSet/samsum-train.csv")
val_data = pd.read_csv(r"DataSet/samsum-validation.csv")

In [4]:
train_data.sample(3)

,id,dialogue,summary
14113,13730872,Matt: Did you get a chance to listen to my new...,"Cam will listen to Matt's new podcast, where h..."
7018,13612261,"Terry: Hi Karen, we srsly need to plan that b-...",Terry and Karen are planning b-day party for T...
2343,13594124-1,Anna: Hi. I'll be late. Sorry!\r\nNina: Ok. Wa...,Anna is 20 min late. Nina will wait.


In [5]:
train_data.shape  

(14732, 3)

In [6]:
val_data.shape

(818, 3)

#### So Train data is large and we dont have more resources so we train us model on random 4000 traing set and validate on 500 random traing set

In [7]:
# random sampleling
train_data = train_data.sample(n=4000 , random_state=42).reset_index(drop= True)
val_data = val_data.sample(n=500 , random_state=42).reset_index(drop= True)


In [8]:
train_data.shape

(4000, 3)

# Data Preprocessing using re

In [3]:
#re :- regular Expression lib
import re

def clean_data(text):
    text = re.sub(r"\r\n"," ",text) # remove \n lines that kind of simble
    text = re.sub(r"\s+"," ",text) # remove spaces
    text = re.sub(r"<.*?>"," ",text) # remove HTML tag ex:- <h1>, <b> 
    text = text.strip().lower()
    return text

In [10]:
# cleaning the training_Data
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)
# cleaning the validation data
val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [11]:
train_data["dialogue"][0] # remove the extra spaces , HTML , \n symbol and convert into lower case 

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

 # Tokenizer

#### "I Love Python" ======> [123,213,324]

###### after apply tokenizer in simple way

In [12]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")  # Their are multiple T5 Model T5-small, t5-base WE use  T5-small

In [13]:
# row data = Tokenize input for fine-Tuning
def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True) 
     
    targets = tokenizer(data["summary"], padding="max_length", max_length=150,truncation=True)

    inputs["labels"] = targets["input_ids"] # token id add into input variable Column name as labels
    return inputs

In [14]:
train_dataset = train_data.apply(tokenize, axis=1).tolist() # apply funtion to tokenize and output convert into list
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [15]:
train_dataset[0] # all are convert into Tokens

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

#### In above there are three section : input_ids,attention_mask,labels
##### 1.IN  input_ids we give the size of 512 so input_ids length is 512 in that first are token remaing like '0' is the padiing upto 512
##### 2.IN attention_mask  in that 1 and 0 so 1 is denoted one is present only that place have token in input_ids ex [25208, 10, 7102,0,0,0] in input_ids so attention_mask [1,1,1,0,0]
##### 3.IN labels same as a input_ids

##### 4.IN input_ids and labels 1= EOS (End Of Sequence) ,0 = Padding

In [16]:
# padding="max_length" → fill empty space with [PAD] until max_length
# max_length=512       → longest size allowed for tokens
# truncation=True      → cut off extra tokens beyond max_length
print("input_ids Length",len(train_dataset[0]["input_ids"]))
print("attention_mask Length",len(train_dataset[0]["attention_mask"]))
print("labels Length",len(train_dataset[0]["labels"]))
print("ALL are List")

input_ids Length 512
attention_mask Length 512
labels Length 150
ALL are List


# Working With Us Model

In [17]:
#NLP => Genration Task
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### 1.Define The GPU

In [7]:
import torch

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    device = "xpu"
elif hasattr(torch.backends, "hip") and torch.backends.hip .is_available():
    device = "hip"
else:
    device = "cpu"

print("Device :-) " ,device)
#model.to(device)

Device :-)  xpu


### 2.Training Argument

In [23]:
training_args = TrainingArguments(
    output_dir = "./result",          # where to save output
    num_train_epochs = 6,             # how many times to train
    weight_decay = 0.01,              # reduce overfitting (small penalty)

    per_device_train_batch_size = 8,  # train batch size per GPU/CPU
    per_device_eval_batch_size = 8,   # eval batch size per GPU/CPU

    eval_strategy = "epoch",          # test after each epoch
    save_strategy = "epoch",          # save after each epoch

    warmup_steps = 500                # slow start for learning rate
)


### 3.Define Traning

In [22]:
trainer = Trainer(
    model = model,               # the model to train
    args = training_args,        # training settings
    train_dataset = train_dataset, # data for training
    eval_dataset = val_dataset     # data for testing/validation
)


### 4.Train Model

In [21]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.569787,0.379768
2,0.393869,0.358396
3,0.369455,0.352319
4,0.358853,0.349030
5,0.351937,0.348242
6,0.347037,0.347857


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.8984897003173828, metrics={'train_runtime': 1938.595, 'train_samples_per_second': 12.38, 'train_steps_per_second': 1.548, 'total_flos': 3248203235328000.0, 'train_loss': 0.8984897003173828, 'epoch': 6.0})

### 5.Save the Model

In [27]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

### 6.Read This Save Model For Further Process ex.Testing

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# Test Core Logic Of Summerization

In [5]:
def summerize_dialogue(dialogue):
    dialogue = clean_data(dialogue) ## clean the text remove space , html tag ,to lowercase
    
    #tokenize
    inputs = tokenizer(
        dialogue,padding="max_length",
        max_length=512 ,       # limit words to 512
        truncation=True,       # cut extra words
        return_tensors="pt"  # make tensor for PyTorch
    ).to(device)

    # genrate the summary ==> token ids
    model.to(device)
    targets = model.generate(

        input_ids = inputs["input_ids"],            # input text ids
        attention_mask = inputs["attention_mask"],   # focus on real words
        max_length= 150,
        num_beams = 4 ,# Transformer genrate 4 output (summary) comapare four and give to us best summary
        early_stopping =True                 # stop when done
    )

    # decode the output  tokenization ===> Readable Word ex:- [213,32424,4232] =====> "I love Python"
    summary = tokenizer.decode(targets[0],skip_special_tokens=True)#EOS (end of sequence), start this are special token that token is skip
    return summary

In [8]:
text_dialogue = """
Durvesh: Bro, I tried cooking yesterday… the smoke alarm clapped louder than my dish!
Sumit: Haha! So basically your cooking got a standing ovation from the fire brigade?
Durvesh: Exactly! Even my neighbors thought I was hosting a rock concert.
Sumit: Next time just invite me, I’ll bring popcorn and a fire extinguisher.
Durvesh: Deal! But you’re washing the dishes after the concert.
Sumit: Only if you promise not to burn water again!
"""

summary = summerize_dialogue(text_dialogue)
print("summary:-",summary)

summary:- sumit tried cooking yesterday, the smoke alarm clapped louder than her dish. she will bring popcorn and a fire extinguisher.


In [57]:
]text_dialogue = input("Enter the Dialogue")
summary = summerize_dialogue(text_dialogue)
print("summary:-",summary)

Enter the Dialogue Durvesh: आज सुबह जिम गया था… लेकिन ट्रेडमिल ने मुझे देखकर खुद ही बंद हो गई! Sumit: हाहा! मतलब मशीन भी तेरी फिटनेस से डर गई? Durvesh: बिल्कुल! उसने कहा 'भाई, पहले आराम कर ले'. Sumit: सही है, अगली बार मैं तेरे साथ आऊँगा… ताकि ट्रेडमिल को लगे कि असली वर्कआउट शुरू हुआ है. Durvesh: तब तो जिम वाले हमें मेंबरशिप फ्री में दे देंगे! Sumit: हाँ, लेकिन शर्त ये होगी कि तू डम्बल्स को सिर्फ फोटो खिंचवाने के लिए उठाएगा.


summary:-       .
